# MGS-25 : Whale Optimization Algorithm — spirale, variante convexe et port de contrôle

**Navigation** : [<< MGS-24 (SA vs mealpy)](MGS-24-SimulatedAnnealing-vs-Mealpy.ipynb) | [Index](README.md)

**Kernel** : Rust (`evcxr`) — deux implémentations indépendantes dans la même exécution

***

## Introduction

Trois paires mesurées, trois visages de l'écart : PSO dominé par mealpy (MGS-22), DE quasi-ex-aequo
en qualité avec vitesse inversée (MGS-23), SA doublé qualité-vitesse par MGS (MGS-24). La paire
**Whale Optimization Algorithm** apporte deux choses neuves à cette série comparative :

- **c'est la paire la plus canonique depuis DE** : les deux implémentations suivent Mirjalili &
  Lewis (2016) avec le même refroidissement linéaire `a = 2 → 0` et le même partage 50/50
  encerclement/spirale — aucun paramètre propre à aligner, le confond paramétrique est quasi nul ;
- **le noyau Rust expose une seconde variante du même moteur** : `Bubble::Convexe`, dont l'opérateur
  bubble-net spiral est remplacé par une simple combinaison convexe `0,5·X + 0,5·X*`. C'est la
  simplification que MGS-19 démontait pièce par pièce ; la mesurer ici face au WOA canonique
  dit si la *forme* de l'opérateur spiral porte de la qualité ou si la moyenne suffit.

Le croisement a donc trois colonnes : **WOA spirale** (noyau), **WOA naive** (noyau, convexe) et
**WOA de contrôle** (port, spirale). Même substrat, budgets mesurés, mêmes graines explicites.

**Ce que Rust apporte** (Mesure, sections 1 et 2) : le protocole apparié, ici porté par deux variantes du même moteur en Rust — 8 000 évaluations par variante du noyau contre 8 050 pour le port, qui évalue sa population initiale en plus des epochs ; la contre-vérification croisée du meilleur candidat du port, redécodé et re-costé côté noyau, qui rend `IDENTIQUE` ; puis le croisement à trois bras sur quatre graines et trois répétitions, déterminisme 12/12, médianes de conflits 50,5 / 49,5 / 50,5, écart maximal à la source de 6 conflits. Le témoin de la source donnait 54 conflits au même budget : le score du port suit son propre flux aléatoire — la fonction de coût reste ce qui est comparé.

***

## 1. Le protocole apparié, pré-enregistré — hérité de MGS-22

Le protocole est celui de MGS-22, adapté à deux implémentations Rust :

- **même substrat** : grille Easy[0] de Sudoku_Easy51.txt, représentation R1 (continu + arrondi), fonction de coût = conflits totaux d'une grille pleine ;
- **budget d'évaluations mesuré, pas supposé** : population 50, avec 160 populations évaluées côté noyau et 160 mises à jour après l'évaluation initiale côté port, soit respectivement 8 000 et 8 050 appels de fitness ;
- **4 graines nommées {0, 1, 7, 42}**, médiane + min/max, jamais un run isolé ;
- **contre-vérification croisée du coût** : le vainqueur du port est décodé et coûté côté noyau — sans elle on risque de comparer deux fonctions de coût, pas deux moteurs ;
- **ms/éval séparé du temps total** ;
- **graines passées explicitement aux deux moteurs** : `StdRng::seed_from_u64(seed)` avant la création de la population ;
- **déterminisme vérifié** (répétition, conflits identiques exigés) avant de publier le moindre chiffre.

**Paramètres de chaque moteur, déclarés avant la mesure** :

| moteur | équations | refroidissement | opérateur bubble-net |
|---|---|---|---|
| WOA spirale (noyau) | Mirjalili & Lewis 2016, Eq. (2.3)-(2.5) | a : 2 → 0 linéaire | spirale `\|X*−X\|·e^{bl}·cos(2πl) + X*`, b = 1, l uniforme dans [−1, 1) |
| WOA naive (noyau) | idem | idem | **combinaison convexe `0,5·X + 0,5·X*`** |
| WOA de contrôle (port) | Mirjalili & Lewis 2016 | a : 2 → 0 linéaire | spirale canonique |

Les deux WOA canoniques partagent donc les équations et les bornes du refroidissement, mais pas la boucle de mise à jour ni la convention de comptage de la population initiale. La colonne naive, elle, isole la question de l'opérateur au sein du noyau : c'est l'exercice 2 qui la pousse plus loin (`mix_coef` ≠ 0,5).

In [2]:
:dep ndarray = "=0.17.2"
:dep rand = "=0.8.5"

{
    // === MGS-25 : socle commun — grille de référence et fonction de coût ===
    // Même socle que MGS-22/23/24 : la représentation R1 est le substrat du banc.
    // Pas de référence à GeneticSharp.Infrastructure.Framework.dll : le RNG est un StdRng local.
    // Pas de référence à MetaGeneticSharp.Infrastructure.dll : le budget est compté en Rust.
    // Pas de référence à MetaGeneticSharp.Domain.dll : la grille et le WOA sont définis dans le carnet.
    use ndarray::Array2;

    // Grille facile Easy[0] de Sudoku_Easy51.txt ; la longueur est vérifiée à la compilation.
    const PUZZLE_LINE_25: &[u8; 81] = b"902005403100063025508407060026309001057010290090670530240530600705200304080041950";

    fn parse_puzzle_25() -> Array2<i64> {
        Array2::from_shape_fn((9, 9), |(r, c)| i64::from(PUZZLE_LINE_25[9 * r + c] - b'0'))
    }

    // Conflits totaux : chaque répétition d'un chiffre dans une ligne, une colonne ou un bloc compte pour un.
    fn count_conflicts_25(g: &Array2<i64>) -> usize {
        let mut conflicts = 0;
        for i in 0..9 {
            let mut seen = [0u16; 3];
            for j in 0..9 {
                let br = 3 * (i / 3) + j / 3;
                let bc = 3 * (i % 3) + j % 3;
                for (mask, value) in seen.iter_mut().zip([g[[i, j]], g[[j, i]], g[[br, bc]]]) {
                    let bit = 1u16 << (value as u32);
                    conflicts += usize::from((*mask & bit) != 0);
                    *mask |= bit;
                }
            }
        }
        conflicts
    }

    fn count_empty_25(p: &Array2<i64>) -> usize {
        p.iter().filter(|&&v| v == 0).count()
    }

    fn empty_cells_25(p: &Array2<i64>) -> Vec<(usize, usize)> {
        p.indexed_iter().filter_map(|(rc, &v)| (v == 0).then_some(rc)).collect()
    }

    // Décodage R1 : arrondi au pair, puis bornage entre 1 et 9, en ordre de lecture.
    fn decode_r1_25(p: &Array2<i64>, empties: &[(usize, usize)], genes: &[f64]) -> Array2<i64> {
        assert_eq!(genes.len(), empties.len());
        let mut g = p.clone();
        for (&(r, c), &v) in empties.iter().zip(genes) {
            g[[r, c]] = v.round_ties_even().clamp(1.0, 9.0) as i64;
        }
        g
    }

    assert!(PUZZLE_LINE_25.iter().all(|b| b.is_ascii_digit()));
    let puzzle = parse_puzzle_25();
    let empties = empty_cells_25(&puzzle);
    let n = count_empty_25(&puzzle);
    let fixed = puzzle.iter().filter(|&&v| v != 0).count();
    assert_eq!((n, fixed, empties.len()), (36, 45, 36));

    // Une demi-valeur teste l'arrondi sans introduire de hasard.
    let g = decode_r1_25(&puzzle, &empties, &vec![2.5; empties.len()]);
    for (rc, &digit) in puzzle.indexed_iter() {
        assert_eq!(g[rc], if digit == 0 { 2 } else { digit });
    }
    assert_eq!(count_conflicts_25(&Array2::from_elem((9, 9), 1)), 216);
    println!("Grille de référence : {n} cellules vides, {fixed} indices fixes, {} gènes R1.", empties.len());
};

Grille de référence : 36 cellules vides, 45 indices fixes, 36 gènes R1.


**Lecture.** Le socle est posé, identique à MGS-22/23/24 au nom près — c'est voulu : la comparabilité
tient à ce que chaque paire courre sur exactement le même substrat. 36 cases
vides = 36 gènes continus dans [1, 10), la fonction de coût compte les doublons ligne/colonne/bloc
d'une grille pleine et vaut 0 ssi résolue. Le décodage parcourt les cases vides dans l'ordre de lecture : il arrondit au pair en cas d'égalité, puis borne le chiffre entre 1 et 9.

In [3]:
// === Moteur WOA : chromosome R1, fitness instrumentée, spirale et variante naive ===
// WOA canonique Mirjalili & Lewis (2016) : a décroît de 2 à 0.
// p < 0,5 : encerclement (|A| > 1 : cible aléatoire, sinon meilleure) ;
// p >= 0,5 : spirale ou, pour la variante naive, combinaison convexe.
{
    use ndarray::Array2;
    use rand::{rngs::StdRng, Rng, SeedableRng};
    use std::{f64::consts::TAU, time::Instant};

    const PUZZLE: &str = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    const DIM: usize = 36;
    const POP: usize = 50;
    const GENERATIONS: usize = 160;
    const EXPECTED_EVALS: usize = 8_000;

    // Même fitness que dans le banc : décodage dans un clone de l'Array2,
    // puis comptage des doublons par masques, comme dans la mesure isolée.
    fn kernel_cost(puzzle: &Array2<i64>, empties: &[(usize, usize)], genes: &[f64]) -> usize {
        let mut grid = puzzle.clone();
        for (&rc, &v) in empties.iter().zip(genes) {
            grid[rc] = v.round_ties_even().clamp(1.0, 9.0) as i64;
        }
        let mut conflicts = 0;
        for i in 0..9 {
            let mut seen = [0u16; 3];
            for j in 0..9 {
                let br = 3 * (i / 3) + j / 3;
                let bc = 3 * (i % 3) + j % 3;
                for (mask, value) in seen.iter_mut().zip([grid[[i, j]], grid[[j, i]], grid[[br, bc]]]) {
                    let bit = 1u16 << (value as u32);
                    conflicts += usize::from(*mask & bit != 0);
                    *mask |= bit;
                }
            }
        }
        conflicts
    }

    #[derive(Clone, Copy)]
    enum Bubble {
        Spirale,
        Convexe { mix_coef: f64 },
    }

    struct Run25 {
        conflicts: usize,
        initial_best: usize,
        evals: usize,
        ms: f64,
    }

    fn run_woa(seed: u64, generations: usize, bubble: Bubble) -> Result<Run25, &'static str> {
        if generations < 3 {
            return Err("Il faut au moins trois générations.");
        }
        let mix_coef = match bubble {
            Bubble::Spirale => None,
            Bubble::Convexe { mix_coef } if (0.0..=1.0).contains(&mix_coef) => Some(mix_coef),
            Bubble::Convexe { .. } => return Err("mix_coef doit appartenir à [0, 1]."),
        };

        // Semer avant la création de la population, comme dans le banc.
        let start = Instant::now();
        let mut rng = StdRng::seed_from_u64(seed);
        let puzzle = Array2::from_shape_fn((9, 9), |(r, c)| i64::from(PUZZLE.as_bytes()[9 * r + c] - b'0'));
        let empties: Vec<_> = puzzle.indexed_iter().filter_map(|(rc, &v)| (v == 0).then_some(rc)).collect();
        if empties.len() != DIM {
            return Err("Nombre de cases vides incompatible avec R1.");
        }
        let mut evals = 0;
        let mut score = |genes: &[f64]| {
            evals += 1;
            kernel_cost(&puzzle, &empties, genes)
        };
        let mut pop: Vec<Vec<f64>> = (0..POP).map(|_| {
            (0..DIM).map(|_| rng.gen_range(1.0..10.0)).collect()
        }).collect();
        let mut best = pop[0].clone();
        let mut best_score = usize::MAX;
        for x in &pop {
            let conflicts = score(x);
            if conflicts < best_score {
                best.clone_from(x);
                best_score = conflicts;
            }
        }
        let initial_best = best_score;

        // Même boucle synchrone, mêmes allocations et même ordre de tirage que dans le banc.
        for step in 0..(generations - 1) {
            let a = 2.0 * (1.0 - step as f64 / (generations - 2) as f64);
            let mut next = Vec::<Vec<f64>>::with_capacity(POP);
            for x in &pop {
                let p = rng.gen_range(0.0_f64..1.0);
                let r1 = rng.gen_range(0.0_f64..1.0);
                let r2 = rng.gen_range(0.0_f64..1.0);
                let l = rng.gen_range(-1.0_f64..1.0);
                let alpha = 2.0 * a * r1 - a;
                let c = 2.0 * r2;
                let genes = if p < 0.5 {
                    let target = if alpha.abs() > 1.0 { &pop[rng.gen_range(0..POP)] } else { &best };
                    x.iter().zip(target).map(|(&v, &t)| (t - alpha * (c * t - v).abs()).clamp(1.0, 10.0)).collect()
                } else {
                    x.iter().zip(&best).map(|(&v, &b)| {
                        let candidate = match mix_coef {
                            None => b + (b - v).abs() * l.exp() * (TAU * l).cos(),
                            Some(w) => w * v + (1.0 - w) * b,
                        };
                        candidate.clamp(1.0, 10.0)
                    }).collect()
                };
                next.push(genes);
            }
            for genes in &next {
                let conflicts = score(genes);
                if conflicts < best_score {
                    best.clone_from(genes);
                    best_score = conflicts;
                }
            }
            pop = next;
        }
        Ok(Run25 { conflicts: best_score, initial_best, evals, ms: start.elapsed().as_secs_f64() * 1_000.0 })
    }

    // Échauffement (courses jetées), puis courses témoins graine 7 : standard ET naive.
    for (label, bubble) in [("spirale", Bubble::Spirale), ("naive convexe", Bubble::Convexe { mix_coef: 0.5 })] {
        let _ = run_woa(123, 10, bubble).expect("Échec de l'échauffement WOA");
        let run = run_woa(7, GENERATIONS, bubble).expect("Échec de la course témoin WOA");
        assert_eq!(run.evals, EXPECTED_EVALS, "Budget du témoin {label}");
        assert!(run.conflicts <= run.initial_best, "Le meilleur mémorisé ne doit pas se dégrader.");
        println!("Noyau WOA {label} (graine 7, témoin) : {} conflits, {} évaluations, {:.0} ms.", run.conflicts, run.evals, run.ms);
    }
};

Noyau WOA spirale (graine 7, témoin) : 49 conflits, 8000 évaluations, 14 ms.


Noyau WOA naive convexe (graine 7, témoin) : 48 conflits, 8000 évaluations, 13 ms.


**Lecture.** Les deux variantes du noyau sont branchées sur le même harnais que les paires précédentes :
chromosome R1, fitness comptée, seeding avant création de population. Les courses témoins
graine 7 donnent les premiers chiffres : la cellule affiche les conflits de chaque variante pour
8 000 évaluations chacune — l'échauffement les précède et n'entre pas dans la mesure. Semer après
la création de la population laisserait les premiers tirages hors du contrôle de la graine.

**Note de parité.** Dans le carnet C# d'origine, les témoins donnent 51 conflits pour la spirale et 49 pour la variante naive. Le port recalcule ses propres trajectoires : une même graine ne reproduit pas le flux du générateur C#. Les budgets sont vérifiés à l'identique ; la comparaison des scores à la source porte sur le banc à quatre graines, pas sur l'égalité de cette course isolée.

In [4]:
// === Contrôle croisé : deux implémentations Rust dans le même noyau ===
// Dans la source, le pont PythonNet charge CPython puis mealpy.
// Ici, ni interpréteur externe ni chargeur : les témoins restent dans le noyau Rust.
{
    use ndarray::Array2;
    use std::collections::HashSet;

    // Grille Easy[0] complète, avec le segment 240530600 ; la taille est vérifiée à la compilation.
    const PUZZLE: &[u8; 81] = b"902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    const DIM: usize = 36;
    const EXPECTED: [usize; 3] = [67, 71, 60];
    const ABS_TOL: usize = 0; // Un vecteur fixé donne un coût entier exact.

    // Le problème R1 : décodage et coût écrits indépendamment de chaque côté.
    // Le témoin ne lance aucun solveur : les graines explicites des courses WOA
    // sont distinctes des graines du LCG, et aucun journal de moteur n'intervient ici.
    fn cost_a(genes: &[f64; DIM]) -> usize {
        let mut g = Array2::from_shape_fn((9, 9), |(r, c)| i64::from(PUZZLE[9 * r + c] - b'0'));
        let empties: Vec<_> = g.indexed_iter().filter_map(|(rc, &v)| (v == 0).then_some(rc)).collect();
        assert_eq!(genes.len(), empties.len());
        for (&rc, &v) in empties.iter().zip(genes) {
            g[rc] = v.round_ties_even().clamp(1.0, 9.0) as i64;
        }
        let mut conflicts = 0;
        for i in 0..9 {
            let mut seen = [0u16; 3];
            for j in 0..9 {
                let br = 3 * (i / 3) + j / 3;
                let bc = 3 * (i % 3) + j % 3;
                for (mask, value) in seen.iter_mut().zip([g[[i, j]], g[[j, i]], g[[br, bc]]]) {
                    let bit = 1u16 << (value as u32);
                    conflicts += usize::from((*mask & bit) != 0);
                    *mask |= bit;
                }
            }
        }
        conflicts
    }

    fn cost_b(genes: &[f64; DIM]) -> usize {
        let mut g = [0u8; 81];
        assert_eq!(genes.len(), PUZZLE.iter().filter(|&&b| b == b'0').count());
        let mut k = 0;
        for (slot, &digit) in g.iter_mut().zip(PUZZLE.iter()) {
            *slot = if digit == b'0' {
                // Arrondi au pair explicite, indépendant de round_ties_even.
                let x = genes[k];
                k += 1;
                let lo = x.floor();
                let fraction = x - lo;
                let up = fraction > 0.5 || (fraction == 0.5 && lo.rem_euclid(2.0) == 1.0);
                (lo + if up { 1.0 } else { 0.0 }).clamp(1.0, 9.0) as u8
            } else {
                digit - b'0'
            };
        }
        let rows: usize = g.chunks_exact(9).map(|row| {
            9 - row.iter().copied().collect::<HashSet<_>>().len()
        }).sum();
        let cols: usize = (0..9).map(|c| {
            9 - g.iter().skip(c).step_by(9).copied().collect::<HashSet<_>>().len()
        }).sum();
        let mut blocks = 0;
        for top in [0, 3, 6] {
            for left in [0, 3, 6] {
                let mut digits = HashSet::new();
                for row in g.chunks_exact(9).skip(top).take(3) {
                    digits.extend(row[left..left + 3].iter().copied());
                }
                blocks += 9 - digits.len();
            }
        }
        rows + cols + blocks
    }

    // --- Sanity check : la fonction de coût est-elle la même sur les témoins ? ---
    // Trois vecteurs témoins déterministes (LCG écrit à la main), décodés et coûtés des deux côtés.
    fn lcg_vector_25(mut state: u32) -> [f64; DIM] {
        std::array::from_fn(|_| {
            state = state.wrapping_mul(1_664_525).wrapping_add(1_013_904_223);
            1.0 + f64::from(state) / 4_294_967_296.0 * 9.0
        })
    }

    assert!(PUZZLE.iter().all(|b| b.is_ascii_digit()));
    let witnesses = [1, 2, 3].map(lcg_vector_25);
    let a: [usize; 3] = std::array::from_fn(|i| cost_a(&witnesses[i]));
    let b: [usize; 3] = std::array::from_fn(|i| cost_b(&witnesses[i]));
    let csv = |xs: &[usize]| xs.iter().map(|v| v.to_string()).collect::<Vec<_>>().join(",");
    let verdict = if a == b { "IDENTIQUE" } else { "DIFFERENT" };
    println!("Sanity check cout : Rust A {} | Rust B {} -> {verdict}", csv(&a), csv(&b));
    for (i, &reference) in EXPECTED.iter().enumerate() {
        assert_eq!(a[i], b[i], "Désaccord A/B sur la graine {}", i + 1);
        assert!(a[i].abs_diff(reference) <= ABS_TOL && b[i].abs_diff(reference) <= ABS_TOL,
                "Graine {} : A = {}, B = {}, attendu = {reference}", i + 1, a[i], b[i]);
    }
};

Sanity check cout : Rust A 67,71,60 | Rust B 67,71,60 -> IDENTIQUE


**Lecture.** Le contrôle est interne à Rust et la **sanity check porte tout le bench** : les
trois vecteurs témoins LCG, décodés et coûtés indépendamment des deux côtés, donnent exactement
les mêmes conflits (67, 71, 60 des deux côtés — `IDENTIQUE`). Sans cet accord, une
différence mesurée entre moteurs pourrait n'être qu'une différence entre les deux fonctions de
coût. Le carnet d'origine croise C# et Python via PythonNet ; ici, le contrôle croise deux implémentations Rust seulement. Il vérifie l'accord sur les témoins, pas l'équivalence sur tous les vecteurs.

In [5]:
// WOA de contrôle : course témoin et re-coût indépendant du vainqueur
{
    use ndarray::Array2;
    use rand::{rngs::StdRng, Rng, SeedableRng};
    use std::{collections::HashSet, f64::consts::TAU, time::Instant};

    const PUZZLE: &str = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    const DIM: usize = 36;
    const POP: usize = 50;
    const EPOCHS: usize = 160;
    const EXPECTED_EVALS: usize = 8_050;
    type Genes = [f64; DIM];

    // Port de contrôle : grille à plat, arrondi explicite et cardinalité des unités.
    fn control_cost(genes: &Genes) -> usize {
        let mut grid = [0u8; 81];
        let mut k = 0;
        for (slot, byte) in grid.iter_mut().zip(PUZZLE.bytes()) {
            *slot = if byte == b'0' {
                let x = genes[k];
                k += 1;
                let lo = x.floor();
                let fraction = x - lo;
                let up = fraction > 0.5 || (fraction == 0.5 && lo.rem_euclid(2.0) == 1.0);
                (lo + if up { 1.0 } else { 0.0 }).clamp(1.0, 9.0) as u8
            } else {
                byte - b'0'
            };
        }
        assert_eq!(k, DIM);
        let rows: usize = grid.chunks_exact(9).map(|row| {
            9 - row.iter().copied().collect::<HashSet<_>>().len()
        }).sum();
        let cols: usize = (0..9).map(|c| {
            9 - grid.iter().skip(c).step_by(9).copied().collect::<HashSet<_>>().len()
        }).sum();
        let mut blocks = 0;
        for top in [0, 3, 6] {
            for left in [0, 3, 6] {
                let mut digits = HashSet::new();
                for row in grid.chunks_exact(9).skip(top).take(3) {
                    digits.extend(row[left..left + 3].iter().copied());
                }
                blocks += 9 - digits.len();
            }
        }
        rows + cols + blocks
    }

    fn run_control(seed: u64, epochs: usize) -> (Genes, usize, usize, f64) {
        assert!(epochs >= 2);
        let start = Instant::now();
        let mut rng = StdRng::seed_from_u64(seed);
        let mut evals = 0;
        let mut score = |genes: &Genes| {
            evals += 1;
            control_cost(genes)
        };
        let mut pop: Vec<(Genes, usize)> = (0..POP).map(|_| {
            let genes = std::array::from_fn(|_| rng.gen_range(1.0..10.0));
            let conflicts = score(&genes);
            (genes, conflicts)
        }).collect();
        let mut best = pop.iter().copied().min_by_key(|entry| entry.1).unwrap_or(pop[0]);

        // La population initiale est évaluée avant les epochs ; chaque epoch
        // propose puis évalue une population entière sans modifier ses cibles en cours de boucle.
        for epoch in 0..epochs {
            let a = 2.0 * (1.0 - epoch as f64 / (epochs - 1) as f64);
            let anchor = best.0;
            let previous = &pop;
            pop = previous.iter().map(|(x, _)| {
                let p = rng.gen_range(0.0_f64..1.0);
                let r1 = rng.gen_range(0.0_f64..1.0);
                let r2 = rng.gen_range(0.0_f64..1.0);
                let l = rng.gen_range(-1.0_f64..1.0);
                let alpha = 2.0 * a * r1 - a;
                let c = 2.0 * r2;
                let next = if p < 0.5 {
                    let target = if alpha.abs() > 1.0 { previous[rng.gen_range(0..POP)].0 } else { anchor };
                    std::array::from_fn(|j| (target[j] - alpha * (c * target[j] - x[j]).abs()).clamp(1.0, 10.0))
                } else {
                    let spiral = l.exp() * (TAU * l).cos();
                    std::array::from_fn(|j| (anchor[j] + (anchor[j] - x[j]).abs() * spiral).clamp(1.0, 10.0))
                };
                let conflicts = score(&next);
                (next, conflicts)
            }).collect();
            for &(genes, conflicts) in &pop {
                if conflicts < best.1 {
                    best = (genes, conflicts);
                }
            }
        }
        (best.0, best.1, evals, start.elapsed().as_secs_f64() * 1_000.0)
    }

    fn kernel_cost(genes: &Genes) -> usize {
        let mut grid = Array2::from_shape_fn((9, 9), |(r, c)| {
            i64::from(PUZZLE.as_bytes()[9 * r + c] - b'0')
        });
        let empties: Vec<_> = grid.indexed_iter().filter_map(|(rc, &v)| (v == 0).then_some(rc)).collect();
        assert_eq!(empties.len(), genes.len());
        for (&rc, &v) in empties.iter().zip(genes) {
            grid[rc] = v.round_ties_even().clamp(1.0, 9.0) as i64;
        }
        let mut conflicts = 0;
        for i in 0..9 {
            let mut seen = [0u16; 3];
            for j in 0..9 {
                let br = 3 * (i / 3) + j / 3;
                let bc = 3 * (i % 3) + j % 3;
                for (mask, value) in seen.iter_mut().zip([grid[[i, j]], grid[[j, i]], grid[[br, bc]]]) {
                    let bit = 1u16 << (value as u32);
                    conflicts += usize::from(*mask & bit != 0);
                    *mask |= bit;
                }
            }
        }
        conflicts
    }

    // Échauffement symétrique (course jetée), puis course témoin graine 7.
    let _ = run_control(123, 10);
    let (genes, conflicts, evals, ms) = run_control(7, EPOCHS);
    assert_eq!(evals, EXPECTED_EVALS, "Budget du témoin de contrôle");
    println!("WOA de contrôle (graine 7, témoin) : {conflicts} conflits, {evals} évaluations, {ms:.0} ms.");

    // Contre-vérification croisée : le vainqueur du port est décodé et coûté par le noyau.
    let checked = kernel_cost(&genes);
    let verdict = if checked == conflicts { "IDENTIQUE" } else { "DIFFERENT" };
    println!("Contre-vérif croisée : coût noyau du meilleur port = {checked} (le port rapporte {conflicts}) -> {verdict}");
    assert_eq!(checked, conflicts, "Coût du vainqueur");
};

WOA de contrôle (graine 7, témoin) : 50 conflits, 8050 évaluations, 55 ms.


Contre-vérif croisée : coût noyau du meilleur port = 50 (le port rapporte 50) -> IDENTIQUE


**Lecture.** La course témoin du port, graine 7, pose le point de comparaison du croisement : le score affiché est obtenu pour **8 050 évaluations**. Et la contre-vérification croisée tient la promesse du protocole : ce meilleur candidat du port, décodé et coûté côté noyau, donne le même nombre de conflits, `IDENTIQUE`. Cet accord vérifie le coût du vainqueur, pas celui de toutes les trajectoires. Dans la source, le témoin mealpy donnait **54 conflits** pour le même budget ; le score du port dépend de son propre flux aléatoire.

***

## 2. Le croisement — trois bras et quatre graines

Population 50, 160 générations (noyau, deux variantes) / 160 epochs (port), graines {0, 1, 7, 42},
trois répétitions par graine et par bras pour la médiane de temps,
déterminisme exigé partout. Le noyau compte l'initialisation dans ses 160 générations ; le port l'évalue avant ses 160 epochs, soit 50 évaluations supplémentaires. La colonne naive court le même budget que la colonne spirale : leur comparaison fait varier l'opérateur à budget et graines constants.

In [6]:
// Banc : trois bras × quatre graines × trois répétitions
// Population 50 × 160 générations/epochs : budgets mesurés par les compteurs.
{
    use ndarray::Array2;
    use rand::{rngs::StdRng, Rng, SeedableRng};
    use std::{collections::HashSet, f64::consts::TAU, time::Instant};

    const PUZZLE: &str = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    const DIM: usize = 36;
    const POP: usize = 50;
    const GENERATIONS: usize = 160;
    const SEEDS: [u64; 4] = [0, 1, 7, 42];
    const LABELS: [&str; 3] = ["WOA spirale", "WOA naive", "WOA de contrôle"];
    const EXPECTED_EVALS: [usize; 3] = [8_000, 8_000, 8_050];
    // Conflits du banc .NET/mealpy d'origine, dans l'ordre des bras et des graines.
    const REFERENCE: [[usize; 4]; 3] = [[46, 50, 51, 49], [53, 50, 49, 46], [52, 50, 54, 50]];
    // Bande de conformité déclarée pour le changement de moteur et de flux RNG.
    const ABS_TOL: usize = 12;
    type Genes = [f64; DIM];
    type Run = (usize, usize, f64);

    // --- Côté noyau, deux variantes : 3 répétitions par graine, ms = médiane ---
    fn kernel_cost(puzzle: &Array2<i64>, empties: &[(usize, usize)], genes: &[f64]) -> usize {
        let mut grid = puzzle.clone();
        for (&rc, &v) in empties.iter().zip(genes) {
            grid[rc] = v.round_ties_even().clamp(1.0, 9.0) as i64;
        }
        let mut conflicts = 0;
        for i in 0..9 {
            let mut seen = [0u16; 3];
            for j in 0..9 {
                let br = 3 * (i / 3) + j / 3;
                let bc = 3 * (i % 3) + j % 3;
                for (mask, value) in seen.iter_mut().zip([grid[[i, j]], grid[[j, i]], grid[[br, bc]]]) {
                    let bit = 1u16 << (value as u32);
                    conflicts += usize::from(*mask & bit != 0);
                    *mask |= bit;
                }
            }
        }
        conflicts
    }

    fn run_kernel(seed: u64, generations: usize, mix_coef: Option<f64>) -> Run {
        assert!(generations >= 3);
        assert!(mix_coef.map_or(true, |w| (0.0..=1.0).contains(&w)));
        let start = Instant::now();
        let mut rng = StdRng::seed_from_u64(seed);
        let puzzle = Array2::from_shape_fn((9, 9), |(r, c)| i64::from(PUZZLE.as_bytes()[9 * r + c] - b'0'));
        let empties: Vec<_> = puzzle.indexed_iter().filter_map(|(rc, &v)| (v == 0).then_some(rc)).collect();
        assert_eq!(empties.len(), DIM);
        let mut evals = 0;
        let mut score = |genes: &[f64]| {
            evals += 1;
            kernel_cost(&puzzle, &empties, genes)
        };
        let mut pop: Vec<Vec<f64>> = (0..POP).map(|_| {
            (0..DIM).map(|_| rng.gen_range(1.0..10.0)).collect()
        }).collect();
        let mut best = pop[0].clone();
        let mut best_score = usize::MAX;
        for x in &pop {
            let conflicts = score(x);
            if conflicts < best_score {
                best.clone_from(x);
                best_score = conflicts;
            }
        }
        // Même calendrier et même ordre de tirage que pour les témoins du noyau.
        for step in 0..(generations - 1) {
            let a = 2.0 * (1.0 - step as f64 / (generations - 2) as f64);
            let mut next = Vec::<Vec<f64>>::with_capacity(POP);
            for x in &pop {
                let p = rng.gen_range(0.0_f64..1.0);
                let r1 = rng.gen_range(0.0_f64..1.0);
                let r2 = rng.gen_range(0.0_f64..1.0);
                let l = rng.gen_range(-1.0_f64..1.0);
                let alpha = 2.0 * a * r1 - a;
                let c = 2.0 * r2;
                let genes = if p < 0.5 {
                    let target = if alpha.abs() > 1.0 { &pop[rng.gen_range(0..POP)] } else { &best };
                    x.iter().zip(target).map(|(&v, &t)| (t - alpha * (c * t - v).abs()).clamp(1.0, 10.0)).collect()
                } else {
                    x.iter().zip(&best).map(|(&v, &b)| {
                        let candidate = match mix_coef {
                            None => b + (b - v).abs() * l.exp() * (TAU * l).cos(),
                            Some(w) => w * v + (1.0 - w) * b,
                        };
                        candidate.clamp(1.0, 10.0)
                    }).collect()
                };
                next.push(genes);
            }
            for genes in &next {
                let conflicts = score(genes);
                if conflicts < best_score {
                    best.clone_from(genes);
                    best_score = conflicts;
                }
            }
            pop = next;
        }
        (best_score, evals, start.elapsed().as_secs_f64() * 1_000.0)
    }

    // --- Côté port : boucle indépendante, population initiale évaluée hors epochs ---
    fn control_cost(genes: &Genes) -> usize {
        let mut grid = [0u8; 81];
        let mut k = 0;
        for (slot, byte) in grid.iter_mut().zip(PUZZLE.bytes()) {
            *slot = if byte == b'0' {
                let x = genes[k];
                k += 1;
                let lo = x.floor();
                let fraction = x - lo;
                let up = fraction > 0.5 || (fraction == 0.5 && lo.rem_euclid(2.0) == 1.0);
                (lo + if up { 1.0 } else { 0.0 }).clamp(1.0, 9.0) as u8
            } else {
                byte - b'0'
            };
        }
        assert_eq!(k, DIM);
        let rows: usize = grid.chunks_exact(9).map(|row| {
            9 - row.iter().copied().collect::<HashSet<_>>().len()
        }).sum();
        let cols: usize = (0..9).map(|c| {
            9 - grid.iter().skip(c).step_by(9).copied().collect::<HashSet<_>>().len()
        }).sum();
        let mut blocks = 0;
        for top in [0, 3, 6] {
            for left in [0, 3, 6] {
                let mut digits = HashSet::new();
                for row in grid.chunks_exact(9).skip(top).take(3) {
                    digits.extend(row[left..left + 3].iter().copied());
                }
                blocks += 9 - digits.len();
            }
        }
        rows + cols + blocks
    }

    fn run_control(seed: u64, epochs: usize) -> Run {
        assert!(epochs >= 2);
        let start = Instant::now();
        let mut rng = StdRng::seed_from_u64(seed);
        let mut evals = 0;
        let mut score = |genes: &Genes| {
            evals += 1;
            control_cost(genes)
        };
        let mut pop: Vec<(Genes, usize)> = (0..POP).map(|_| {
            let genes = std::array::from_fn(|_| rng.gen_range(1.0..10.0));
            let conflicts = score(&genes);
            (genes, conflicts)
        }).collect();
        let mut best = pop.iter().copied().min_by_key(|entry| entry.1).unwrap_or(pop[0]);
        for epoch in 0..epochs {
            let a = 2.0 * (1.0 - epoch as f64 / (epochs - 1) as f64);
            let anchor = best.0;
            let previous = &pop;
            pop = previous.iter().map(|(x, _)| {
                let p = rng.gen_range(0.0_f64..1.0);
                let r1 = rng.gen_range(0.0_f64..1.0);
                let r2 = rng.gen_range(0.0_f64..1.0);
                let l = rng.gen_range(-1.0_f64..1.0);
                let alpha = 2.0 * a * r1 - a;
                let c = 2.0 * r2;
                let next = if p < 0.5 {
                    let target = if alpha.abs() > 1.0 { previous[rng.gen_range(0..POP)].0 } else { anchor };
                    std::array::from_fn(|j| (target[j] - alpha * (c * target[j] - x[j]).abs()).clamp(1.0, 10.0))
                } else {
                    let spiral = l.exp() * (TAU * l).cos();
                    std::array::from_fn(|j| (anchor[j] + (anchor[j] - x[j]).abs() * spiral).clamp(1.0, 10.0))
                };
                let conflicts = score(&next);
                (next, conflicts)
            }).collect();
            for &(genes, conflicts) in &pop {
                if conflicts < best.1 {
                    best = (genes, conflicts);
                }
            }
        }
        (best.1, evals, start.elapsed().as_secs_f64() * 1_000.0)
    }

    fn summary(mut scores: [usize; 4]) -> (f64, usize, usize) {
        scores.sort_unstable();
        ((scores[1] + scores[2]) as f64 / 2.0, scores[0], scores[3])
    }

    // --- Table ---
    let mut conflicts_by_arm = [[0usize; 4]; 3];
    let mut mean_ms_eval = [0.0_f64; 3];
    let mut stable = 0;
    let mut max_gap = 0;
    let mut deltas = [[0usize; 4]; 3];
    println!("{:<17} {:>6} {:>9} {:>7} {:>8} {:>8}", "moteur", "graine", "conflits", "evals", "ms", "ms/eval");
    for (arm, label) in LABELS.iter().enumerate() {
        for (i, &seed) in SEEDS.iter().enumerate() {
            let runs: [Run; 3] = std::array::from_fn(|_| match arm {
                0 => run_kernel(seed, GENERATIONS, None),
                1 => run_kernel(seed, GENERATIONS, Some(0.5)),
                _ => run_control(seed, GENERATIONS),
            });
            let (conflicts, evals, _) = runs[0];
            stable += usize::from(runs.iter().all(|run| run.0 == conflicts));
            assert!(runs.iter().all(|run| run.1 == EXPECTED_EVALS[arm]), "Budget : {label}, graine {seed}");
            let mut times = runs.map(|run| run.2);
            times.sort_by(f64::total_cmp);
            let ms = times[1];
            let ms_eval = ms / evals as f64;
            println!("{label:<17} {seed:>6} {conflicts:>9} {evals:>7} {ms:>8.0} {ms_eval:>8.3}");
            conflicts_by_arm[arm][i] = conflicts;
            mean_ms_eval[arm] += ms_eval / SEEDS.len() as f64;
            let delta = conflicts.abs_diff(REFERENCE[arm][i]);
            deltas[arm][i] = delta;
            max_gap = max_gap.max(delta);
        }
    }
    println!();
    for (arm, label) in LABELS.iter().enumerate() {
        let (med, min, max) = summary(conflicts_by_arm[arm]);
        println!("{label:<17} : médiane conflits {med:.1} (min {min}, max {max}), ms/éval moyen {:.3}", mean_ms_eval[arm]);
    }
    assert!(mean_ms_eval.iter().all(|&t| t.is_finite() && t > 0.0));
    println!("Rapport ms/éval port/noyau spirale : {:.2}x", mean_ms_eval[2] / mean_ms_eval[0]);
    println!("Déterminisme : conflits identiques sur les 3 répétitions pour {stable}/12 paires graine-moteur.");
    assert_eq!(stable, 12);

    // Parité : écarts calculés aux scores stockés dans le carnet .NET, pas aux durées.
    for (arm, label) in LABELS.iter().enumerate() {
        let (med, min, max) = summary(conflicts_by_arm[arm]);
        let (ref_med, ref_min, ref_max) = summary(REFERENCE[arm]);
        let aggregate_gaps = ((med - ref_med).abs(), min.abs_diff(ref_min), max.abs_diff(ref_max));
        let ok = deltas[arm].iter().all(|&d| d <= ABS_TOL)
            && aggregate_gaps.0 <= ABS_TOL as f64 && aggregate_gaps.1 <= ABS_TOL && aggregate_gaps.2 <= ABS_TOL;
        println!("Parité {label} : écarts graines {:?}, médiane {:.1}, min {}, max {} (tolérance {ABS_TOL}) -> {}",
                 deltas[arm], aggregate_gaps.0, aggregate_gaps.1, aggregate_gaps.2, if ok { "CONFORME" } else { "ÉCART" });
        assert!(ok, "Bande de conformité dépassée : {label}");
    }
    let (spiral_wins, naive_wins, ties) = (0..SEEDS.len()).fold((0, 0, 0), |(s, n, t), i| {
        match conflicts_by_arm[0][i].cmp(&conflicts_by_arm[1][i]) {
            std::cmp::Ordering::Less => (s + 1, n, t),
            std::cmp::Ordering::Greater => (s, n + 1, t),
            std::cmp::Ordering::Equal => (s, n, t + 1),
        }
    });
    assert_eq!(spiral_wins + naive_wins + ties, SEEDS.len());
    println!("Comparaison appariée du noyau : {spiral_wins} victoires spirale, {naive_wins} victoires naive, {ties} égalités.");
    println!("Écart maximal à la source parmi les 12 scores : {max_gap} conflits.");
};

moteur            graine  conflits   evals       ms  ms/eval


WOA spirale            0        52    8000       16    0.002


WOA spirale            1        50    8000       17    0.002


WOA spirale            7        49    8000       14    0.002


WOA spirale           42        51    8000       14    0.002


WOA naive              0        50    8000       11    0.001


... (13 outputs left out)

Parité WOA naive : écarts graines [3, 4, 1, 3], médiane 0.0, min 2, max 1 (tolérance 12) -> CONFORME


Parité WOA de contrôle : écarts graines [0, 0, 4, 1], médiane 0.5, min 0, max 2 (tolérance 12) -> CONFORME


Comparaison appariée du noyau : 1 victoires spirale, 3 victoires naive, 0 égalités.


Écart maximal à la source parmi les 12 scores : 6 conflits.


In [7]:
// Fitness seule : 500 vecteurs communs et cinq répétitions par côté
// Vecteurs générés en Rust (LCG, graines 42..541) et rejoués sans sérialisation :
// les deux côtés chronomètrent décodage + coût sur exactement les mêmes points.
{
    use ndarray::Array2;
    use std::{collections::HashSet, hint::black_box, time::Instant};

    const PUZZLE: &str = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    const DIM: usize = 36;
    const K: usize = 500;
    const REPS: usize = 5;
    type Genes = [f64; DIM];

    fn decode_kernel(puzzle: &Array2<i64>, empties: &[(usize, usize)], genes: &Genes) -> Array2<i64> {
        let mut grid = puzzle.clone();
        for (&rc, &v) in empties.iter().zip(genes) {
            grid[rc] = v.round_ties_even().clamp(1.0, 9.0) as i64;
        }
        grid
    }

    fn cost_kernel(grid: &Array2<i64>) -> usize {
        let mut conflicts = 0;
        for i in 0..9 {
            let mut seen = [0u16; 3];
            for j in 0..9 {
                let br = 3 * (i / 3) + j / 3;
                let bc = 3 * (i % 3) + j % 3;
                for (mask, value) in seen.iter_mut().zip([grid[[i, j]], grid[[j, i]], grid[[br, bc]]]) {
                    let bit = 1u16 << (value as u32);
                    conflicts += usize::from(*mask & bit != 0);
                    *mask |= bit;
                }
            }
        }
        conflicts
    }

    fn decode_control(genes: &Genes) -> [u8; 81] {
        let mut grid = [0u8; 81];
        let mut k = 0;
        for (slot, byte) in grid.iter_mut().zip(PUZZLE.bytes()) {
            *slot = if byte == b'0' {
                let x = genes[k];
                k += 1;
                let lo = x.floor();
                let fraction = x - lo;
                let up = fraction > 0.5 || (fraction == 0.5 && lo.rem_euclid(2.0) == 1.0);
                (lo + if up { 1.0 } else { 0.0 }).clamp(1.0, 9.0) as u8
            } else {
                byte - b'0'
            };
        }
        assert_eq!(k, DIM);
        grid
    }

    fn cost_control(grid: &[u8; 81]) -> usize {
        let rows: usize = grid.chunks_exact(9).map(|row| {
            9 - row.iter().copied().collect::<HashSet<_>>().len()
        }).sum();
        let cols: usize = (0..9).map(|c| {
            9 - grid.iter().skip(c).step_by(9).copied().collect::<HashSet<_>>().len()
        }).sum();
        let mut blocks = 0;
        for top in [0, 3, 6] {
            for left in [0, 3, 6] {
                let mut digits = HashSet::new();
                for row in grid.chunks_exact(9).skip(top).take(3) {
                    digits.extend(row[left..left + 3].iter().copied());
                }
                blocks += 9 - digits.len();
            }
        }
        rows + cols + blocks
    }

    fn lcg_vector(mut state: u32) -> Genes {
        std::array::from_fn(|_| {
            state = state.wrapping_mul(1_664_525).wrapping_add(1_013_904_223);
            1.0 + f64::from(state) / 4_294_967_296.0 * 9.0
        })
    }

    fn time_batch(vectors: &[Genes], fitness: impl Fn(&Genes) -> usize) -> (f64, usize) {
        let start = Instant::now();
        let mut total = 0;
        for v in vectors {
            total += black_box(fitness(black_box(v)));
        }
        (start.elapsed().as_secs_f64() * 1_000.0, total)
    }

    let puzzle = Array2::from_shape_fn((9, 9), |(r, c)| i64::from(PUZZLE.as_bytes()[9 * r + c] - b'0'));
    let empties: Vec<_> = puzzle.indexed_iter().filter_map(|(rc, &v)| (v == 0).then_some(rc)).collect();
    assert_eq!(empties.len(), DIM);
    let vectors: Vec<_> = (42..=541).map(lcg_vector).collect();
    assert_eq!(vectors.len(), K);
    let kernel = |v: &Genes| cost_kernel(&decode_kernel(&puzzle, &empties, v));
    let control = |v: &Genes| cost_control(&decode_control(v));

    // Vérification point par point et échauffement, hors chronométrage.
    let mut matches = 0;
    let mut checksum = 0;
    for (i, genes) in vectors.iter().enumerate() {
        let a = kernel(genes);
        let b = control(genes);
        assert_eq!(a, b, "Désaccord pour la graine LCG {}", i + 42);
        matches += usize::from(a == b);
        checksum += a;
    }
    assert_eq!(matches, K);

    // Alterner l'ordre de passage et retenir la médiane de cinq pour chaque côté.
    let mut times_a = [0.0; REPS];
    let mut times_b = [0.0; REPS];
    for rep in 0..REPS {
        let (a, b) = if rep % 2 == 0 {
            (time_batch(&vectors, &kernel), time_batch(&vectors, &control))
        } else {
            let b = time_batch(&vectors, &control);
            (time_batch(&vectors, &kernel), b)
        };
        assert_eq!((a.1, b.1), (checksum, checksum));
        times_a[rep] = a.0;
        times_b[rep] = b.0;
    }
    times_a.sort_by(f64::total_cmp);
    times_b.sort_by(f64::total_cmp);
    let (a, b) = (times_a[REPS / 2], times_b[REPS / 2]);
    assert!(a.is_finite() && b.is_finite() && a > 0.0 && b > 0.0);
    println!("Fitness seule, {K} vecteurs identiques (médiane de {REPS} répétitions par côté) :");
    println!("  Noyau : {a:.1} ms total -> {:.3} ms/éval", a / K as f64);
    println!("  Port  : {b:.1} ms total -> {:.3} ms/éval", b / K as f64);
    println!("  rapport port/noyau : {:.2}x", b / a);
    let order = match a.total_cmp(&b) {
        std::cmp::Ordering::Less => "noyau plus rapide",
        std::cmp::Ordering::Greater => "port plus rapide",
        std::cmp::Ordering::Equal => "médianes égales",
    };
    println!("  Ordre observé : {order} ; accord des coûts : {matches}/{K}.");
};

Fitness seule, 500 vecteurs identiques (médiane de 5 répétitions par côté) :


  Noyau : 0.7 ms total -> 0.001 ms/éval


  Port  : 3.4 ms total -> 0.007 ms/éval


  rapport port/noyau : 4.91x


  Ordre observé : noyau plus rapide ; accord des coûts : 500/500.


**Lecture du croisement.** Pour départager les bras, lisez ensemble la médiane et l'étendue des conflits, puis revenez aux scores par graine :

- **qualité** : dans le banc Rust, la variante naive affiche la médiane la plus basse, `49.5` (min 48, max 54). La spirale et le port de contrôle affichent tous deux `50.5`, avec des étendues respectives de 49 à 52 et de 50 à 52. Les écarts à la référence sont calculés avec une bande de 12 conflits, qui n'est pas un test d'équivalence entre moteurs ;
- **opérateur** : la ligne « Comparaison appariée du noyau » indique combien de graines favorisent la spirale ou la combinaison convexe, et combien les laissent à égalité. À budget et graines constants, cette comparaison renseigne sur le choix de l'opérateur dans le noyau, pas sur tous les Sudoku ni sur tous les budgets ;
- **vitesse** : le ratio est calculé sur les valeurs non arrondies. Le ratio **moteur** est lui aussi sensible à la machine et à la charge, pas seulement le ratio fitness. Le banc mesure le moteur complet ; la mesure isolée compare le décodage et le comptage sur les mêmes 500 vecteurs. L'ordre de vitesse à retenir est celui indiqué dans la sortie, pas un classement garanti ;
- **protocole** : 8 000 évaluations pour chaque variante du noyau contre 8 050 pour le port, qui évalue sa population initiale en plus des epochs. Les compteurs du banc vérifient le déterminisme sur les 12 paires, et le re-coût du vainqueur du port est contrôlé séparément.

Dans la série d'origine, les médianes du WOA étaient plus élevées que celles du SA, du DE et du PSO sur ce problème. La contraction de la population est une explication possible, mais ce banc ne mesure pas directement la diversité.

**Lecture du coût par évaluation.** Le contrôle point par point élimine un désaccord de coût sur les vecteurs chronométrés. L'écart de temps mesuré entre les deux implémentations Rust ne mesure pas un effet de langage : il comprend notamment la différence entre les masques de bits et les ensembles de chiffres, ainsi que le coût de leurs décodages.

***

## Exercice 1 : budget ×4 — l'ordre des trois colonnes survit-il ?

MGS-22/23/24 posaient la même question. Un écart qui se referme à budget accru dit « le moteur
distillé converge plus lentement mais atteint le même plateau » ; un écart stable dit « plateau
différent ». Ici la question a une dimension supplémentaire : la colonne naive, dont l'opérateur
convexe ramène le candidat vers le meilleur courant, a-t-elle un déficit que
le budget comble — ou est-ce structurel ?

```text
À compléter (décommentez dans la cellule suivante) :
1. Relancez les trois bras, à population 50 et budget nominal ×4 (noyau : 640 générations, les deux variantes ; port : 640 epochs).
2. Comparez les médianes obtenues à celles du croisement.
3. Verdict : l'ordre spirale/naive/port est-il stable à budget accru ?
```

In [8]:
// EXERCICE 1 : budget x4 — population 50, 640 générations (spirale + naive) ; port 640 epochs.
// Dans une copie du banc, gardez les fonctions des moteurs et remplacez le bloc de mesure par ce cadre à compléter.
// const BUDGET: usize = 640;
// for seed in SEEDS {
//     let spirale = run_kernel(seed, BUDGET, None);
//     let naive = run_kernel(seed, BUDGET, Some(0.5));
//     let port = run_control(seed, BUDGET);
//     println!("WOA x4 (graine {seed}) : spirale {} conflits ({} évaluations, {:.0} ms), naive {} conflits ({} évaluations, {:.0} ms), port {} conflits ({} évaluations, {:.0} ms).",
//         spirale.0, spirale.1, spirale.2, naive.0, naive.1, naive.2, port.0, port.1, port.2);
// }
// TODO étudiant : calculer les médianes par bras et comparer leur classement à celui du banc.
println!("Exercice a completer (decommentez le bloc ci-dessus).");

Exercice a completer (decommentez le bloc ci-dessus).


## Exercice 2 : `mix_coef` — de combien penche la moyenne du Naive ?

L'opérateur naive écrit `mix_coef·X + (1−mix_coef)·X*` avec `mix_coef` = 0,5 : le candidat et la
meilleure solution reçoivent le même poids. `run_kernel` accepte un `mix_coef` libre — pencher la moyenne vers X* (0,25) ou vers
le candidat (0,75) dit dans quel sens l'asymétrie aide. Comparez les deux poids sur les quatre graines du banc, au budget standard.

In [9]:
// Exercice 2 — tester deux coefficients de mélange
// mix_coef : 0.25 vers X*, 0.75 vers le candidat ; quatre graines, budget standard.
// Dans une copie du banc, gardez les fonctions des moteurs et remplacez le bloc de mesure par ce cadre à compléter.
// for mix_coef in [0.25, 0.75] {
//     let mut serie = Vec::with_capacity(SEEDS.len());
//     for seed in SEEDS {
//         let (conflits, _, _) = run_kernel(seed, GENERATIONS, Some(mix_coef));
//         serie.push(format!("graine {seed}: {conflits} conflits"));
//     }
//     println!("mix_coef={mix_coef:.2} : {}", serie.join(" | "));
// }
// TODO étudiant : comparer les deux séries graine par graine et interpréter les écarts.
println!("Exercice a completer (decommentez le bloc ci-dessus).");

Exercice a completer (decommentez le bloc ci-dessus).


## Exercice 3 : profiler la fitness — où va la milliseconde ?

La cellule du coût par évaluation compare déjà le total décodage + coût. Pour localiser la
différence, séparez les deux étapes côté port (décoder une fois, coûter N fois) et comparez
au profil équivalent côté noyau, sur les mêmes 500 vecteurs.

In [10]:
// Exercice 3 — isoler les deux étapes de la fitness
// Profil décodage vs coût, 500 vecteurs, deux côtés (adapté de MGS-23/24, exercice 3).
// Complétez ce cadre dans une copie de la cellule « Fitness seule », après la création de vectors.
// fn profiler<G>(vectors: &[Genes], decoder: impl Fn(&Genes) -> G, couter: impl Fn(&G) -> usize) -> (f64, f64) {
//     let debut_decodage = Instant::now();
//     let grilles: Vec<G> = vectors.iter().map(|v| decoder(black_box(v))).collect();
//     let duree_decodage = debut_decodage.elapsed();
//
//     let debut_cout = Instant::now();
//     for grille in &grilles {
//         black_box(couter(black_box(grille)));
//     }
//     let duree_cout = debut_cout.elapsed();
//     (duree_decodage.as_secs_f64(), duree_cout.as_secs_f64())
// }
// let noyau = profiler(&vectors, |v| decode_kernel(&puzzle, &empties, v), cost_kernel);
// let port = profiler(&vectors, decode_control, cost_control);
// let ms_par_vecteur = 1_000.0 / vectors.len() as f64;
// for (label, (decodage, cout)) in [("Noyau", noyau), ("Port", port)] {
//     println!("{label} : décodage seul {:.3} ms/vec, coût {:.3} ms/vec",
//         decodage * ms_par_vecteur, cout * ms_par_vecteur);
// }
// TODO étudiant : déterminer quelle étape explique l'écart.
println!("Exercice a completer (decommentez le bloc ci-dessus).");

Exercice a completer (decommentez le bloc ci-dessus).


## Références et provenance

- Seyedali Mirjalili et Andrew Lewis (2016), *The Whale Optimization Algorithm*, **Advances in Engineering Software**, 95, 51–67 — équations d'encerclement et de spirale.
- [MGS-22 — MGS contre mealpy](MGS-22-MGS-vs-Mealpy.ipynb) — protocole apparié : même grille, budgets mesurés, graines explicites et contre-vérification du coût.

## À retenir

**Note de parité.** La grille, les bornes de décodage et les graines nommées sont les mêmes que dans le carnet d'origine. `StdRng` ne reproduit pas les flux .NET et NumPy : les trajectoires et les scores stochastiques sont propres au port.

| Section | Contrôle de parité |
|---|---|
| Grille R1 | Écart attendu de 0 sur les trois comptes : 36 cases vides, 45 indices fixes, 36 gènes. |
| Témoins LCG | Écart attendu de 0 aux coûts de référence 67, 71 et 60, dans les deux implémentations. |
| Courses témoins | Écart attendu de 0 sur les budgets : 8 000 évaluations pour chaque variante du noyau et 8 050 pour le port. Le coût annoncé du meilleur candidat du port doit égaler son coût recalculé par le noyau. |
| Banc apparié | Les lignes « Parité » donnent les écarts mesurés pour chaque graine, ainsi que pour la médiane, le minimum et le maximum de chaque bras, sous une tolérance absolue de 12 conflits. Le contrôle de déterminisme exige 12/12 paires graine–moteur stables sur les trois répétitions. |
| Fitness isolée | Accord exact contrôlé sur les 500 vecteurs partagés. Les durées, les rapports et l'ordre de vitesse sont des mesures locales, non des cibles de parité avec la source. |

La médiane et l'étendue décrivent la qualité sur ces quatre graines. Le tableau par graine et le décompte des victoires et égalités permettent de distinguer un écart régulier entre les deux variantes du noyau d'un résultat porté par une seule graine.

## Vérifiez votre compréhension

1. Quel côté gagne en qualité, et de combien ?
   <details><summary>Réponse</summary>

   Le bras à la médiane de conflits la plus basse est en tête sur ce banc ; l'écart se lit en soustrayant les deux médianes comparées. En cas d'égalité, les scores par graine montrent si les bras font jeu égal à chaque course ou si leurs avantages se compensent.

   </details>

2. Quel est l'écart de coût, et d'où vient-il ?
   <details><summary>Réponse</summary>

   Le banc donne le rapport des temps par évaluation du moteur complet, tandis que la fitness isolée donne le rapport pour le décodage et le comptage des conflits seuls. Puisque les deux implémentations tournent en Rust, cet écart n'est pas un effet de langage. L'exercice 3 aide à séparer les deux étapes de la fitness.

   </details>

3. Les deux variantes du noyau se distinguent-elles ?
   <details><summary>Réponse</summary>

   Elles se distinguent par l'opérateur bubble-net : spirale ou combinaison convexe. Pour savoir si cela change la qualité à ce budget, lisez la ligne « Comparaison appariée du noyau ». Une médiane identique ne suffit pas à conclure que les variantes se valent pour chaque graine.

   </details>

4. Que reste-t-il à aligner entre le noyau et le port ?
   <details><summary>Réponse</summary>

   La grille, la taille de population, les graines nommées et les bornes du refroidissement sont communes. Le budget n'est pas strictement égal : le port évalue 50 candidats initiaux en plus de ses epochs. Pour comparer à nombre d'appels strictement égal, il faudrait inclure cette population dans le même budget total pour les deux moteurs.

   </details>

## Pour aller plus loin

**Carnet suivant.** [MGS-26 — EquilibriumOptimizer MGS contre mealpy](MGS-26-EquilibriumOptimizer-vs-Mealpy.ipynb) — la paire suivante du même protocole apparié : un port face à son original, et le plus large écart de qualité en faveur de mealpy de la campagne.

**Lectures.**

- **Le protocole apparié** : hérité de [MGS-22](MGS-22-MGS-vs-Mealpy.ipynb) — même grille, budgets mesurés, mêmes graines, déterminisme vérifié.
- **L'axe Stratégie / Moteur** : comparer les deux opérateurs dans le noyau renseigne sur la stratégie ; comparer les deux spirales renseigne sur les implémentations, avec la réserve des 50 évaluations supplémentaires du port.

**Carnets voisins.**

- **[MGS-24 — SimulatedAnnealing MGS contre mealpy](MGS-24-SimulatedAnnealing-vs-Mealpy.ipynb)** — la paire précédente : deux recuits à aligner.
- **[MGS-31 — Synthèse croisée MGS contre mealpy](MGS-31-Synthese-Croisee.ipynb)** — la synthèse de la campagne : la comparaison WOA d'origine y est replacée parmi les neuf.
- **[README de la sous-série](README.md)** — la question directrice, le protocole commun et la navigation de la sous-série.

> Ce carnet adapte en Rust le protocole du [MGS-25 original](MGS-25-WhaleOptimisation-vs-Mealpy.ipynb), qui compare MGS en C# à mealpy via PythonNet.